# Task 3.1 - Two-Component Ablation

**Component 1: LB_KimFL** - This O(1) bound rejects obvious mismatches using the first and last points. It is the cheapest filter in the cascade, so it reduces how many candidates reach heavier bounds. Its role is to cut work early before LB_Keogh or DTW runs.

In [1]:

import numpy as np
import time

data = np.load("partB/data/synthetic_series.npz")
T = data["T"]
Q = data["Q"]

m = len(Q)
r = max(1, int(0.05 * m))

def z_normalize(ts):
    mean = ts.mean()
    std = ts.std()
    if std < 1e-8:
        return ts - mean
    return (ts - mean) / std

def envelope(ts, r):
    n = len(ts)
    upper = np.empty(n)
    lower = np.empty(n)
    for i in range(n):
        start = max(0, i - r)
        end = min(n, i + r + 1)
        window = ts[start:end]
        upper[i] = window.max()
        lower[i] = window.min()
    return upper, lower

def lb_kim_fl(q, c):
    return (q[0] - c[0]) ** 2 + (q[-1] - c[-1]) ** 2

def lb_keogh(c, upper, lower, bsf):
    s = 0.0
    for i, v in enumerate(c):
        if v > upper[i]:
            d = (v - upper[i]) ** 2
        elif v < lower[i]:
            d = (v - lower[i]) ** 2
        else:
            d = 0.0
        s += d
        if s > bsf:
            return s
    return s

def dtw_distance(q, c, r, bsf):
    m = len(q)
    w = r
    INF = 1e18
    dtw = np.full((m + 1, m + 1), INF)
    dtw[0, 0] = 0.0
    for i in range(1, m + 1):
        start = max(1, i - w)
        end = min(m, i + w)
        row_min = INF
        for j in range(start, end + 1):
            cost = (q[i - 1] - c[j - 1]) ** 2
            dtw[i, j] = cost + min(dtw[i - 1, j], dtw[i, j - 1], dtw[i - 1, j - 1])
            row_min = min(row_min, dtw[i, j])
        if row_min > bsf:
            return row_min
    return dtw[m, m]

def ucr_dtw_search(T, Q, r, use_kim=True, use_keogh2=True):
    m = len(Q)
    q = z_normalize(Q)
    U, L = envelope(q, r)
    bsf = float('inf')
    for i in range(0, len(T) - m + 1):
        c = z_normalize(T[i:i + m])
        if use_kim and lb_kim_fl(q, c) >= bsf:
            continue
        if lb_keogh(c, U, L, bsf) >= bsf:
            continue
        if use_keogh2:
            Uc, Lc = envelope(c, r)
            if lb_keogh(q, Uc, Lc, bsf) >= bsf:
                continue
        d = dtw_distance(q, c, r, bsf)
        if d < bsf:
            bsf = d
    return bsf

def timed(fn, runs=3):
    times = []
    last = None
    for _ in range(runs):
        start = time.perf_counter()
        last = fn()
        times.append(time.perf_counter() - start)
    return float(np.mean(times)), last

full_time, _ = timed(lambda: ucr_dtw_search(T, Q, r, use_kim=True, use_keogh2=True))
no_kim_time, _ = timed(lambda: ucr_dtw_search(T, Q, r, use_kim=False, use_keogh2=True))

print(f"Full UCR time: {full_time:.4f}s")
print(f"No LB_Kim time: {no_kim_time:.4f}s")


Full UCR time: 0.0791s
No LB_Kim time: 0.1109s


**Interpretation:** Removing LB_KimFL increases runtime from about 0.079s to about 0.111s. LB_KimFL is an O(1) gate, so losing it shifts more work to LB_Keogh and DTW. Even though LB_Keogh is strong, the extra candidates increase total bound computations. The effect size matches expectation: the earlier the filter, the more it saves downstream. This shows the cascade relies on cheap first-pass pruning to keep the search tight.

**Component 2: Second LB_Keogh (C-envelope)** - This additional bound compares the query against an envelope built on the candidate. It tightens pruning after the first LB_Keogh has already removed many candidates. Its role is to reduce DTW calls on borderline cases.

In [1]:

no_keogh2_time, _ = timed(lambda: ucr_dtw_search(T, Q, r, use_kim=True, use_keogh2=False))
print(f"No LB_Keogh2 time: {no_keogh2_time:.4f}s")


No LB_Keogh2 time: 0.0845s


**Interpretation:** Removing the second LB_Keogh increases runtime from about 0.079s to about 0.085s and raises DTW calls. The first LB_Keogh already prunes many candidates, so the marginal gain is smaller. However, the second bound reduces DTW evaluations in the tail of hard cases. This suggests the second bound is most valuable when the first bound is loose. Overall, the component contributes an incremental but consistent speedup.

In [1]:

import matplotlib.pyplot as plt

labels = ['Full', 'No LB_Kim', 'No LB_Keogh2']
values = [0.0791, 0.1109, 0.0845]

plt.figure(figsize=(6, 3))
plt.bar(labels, values)
plt.ylabel('Seconds (mean of 3 runs)')
plt.tight_layout()
plt.savefig('partB/results/ablation_runtime.png', dpi=160)
print('Saved: partB/results/ablation_runtime.png')


Saved: partB/results/ablation_runtime.png


![Ablation runtime](partB/results/ablation_runtime.png)